# Group 3 | Seasonal Climatology

**Predicting Drought Before the Rains Fail**

District-level severe meteorological drought, Tanzania (1981–2025).

## 5. Retrospective forecasting, 1–3 months ahead

**Objective.** Reconstruct what the baseline would have forecast at each past issue month, recording the prediction date, target date, numerator, denominator and subsequent observed SPI-3 value.

For target month $T$ and lead $h\in\{1,2,3\}$:

$$\hat p_{d,T\mid T-h}=\frac{D_{d,m(T),\mathrm{past}}}{N_{d,m(T),\mathrm{past}}}.$$

The preceding same-calendar-month observation is at least nine months before the issue month for these forecast leads. This implementation therefore uses historical observations known before each issue date.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
OUT = ROOT / "outputs"
for name in ["figures", "maps", "metrics", "predictions", "tables"]:
    (OUT / name).mkdir(parents=True, exist_ok=True)

import pandas as pd
from IPython.display import display
from group3_climatology.data import load_data
from group3_climatology.baseline import forecast_table, inspect_forecast
from group3_climatology.config import INPUT

data=load_data(INPUT)
forecasts=forecast_table(data)
print(f"Saved forecast records: {len(forecasts):,}")
print("Forecast horizons:",sorted(forecasts.horizon_months.unique()))
display(forecasts.head(9))
forecasts.to_csv(OUT / "predictions" / "seasonal_climatology_forecasts_2015_2025.csv",index=False)

Saved forecast records: 59,400
Forecast horizons: [np.int64(1), np.int64(2), np.int64(3)]


,district_id,region,district,issue_month,target_month,horizon_months,calendar_month,historical_severe_years,historical_valid_years,predicted_probability,observed_spi3,observed_severe,severe_onset,latest_used_month_upper_bound
0,0101,Dodoma,Kondoa,2014-12-01,2015-01-01,1,1,1.0,33.0,0.030303,-0.042829,0,0,2014-01-01
1,0101,Dodoma,Kondoa,2014-11-01,2015-01-01,2,1,1.0,33.0,0.030303,-0.042829,0,0,2014-01-01
2,0101,Dodoma,Kondoa,2014-10-01,2015-01-01,3,1,1.0,33.0,0.030303,-0.042829,0,0,2014-01-01
3,0102,Dodoma,Mpwapwa,2014-12-01,2015-01-01,1,1,1.0,33.0,0.030303,0.103148,0,0,2014-01-01
4,0102,Dodoma,Mpwapwa,2014-11-01,2015-01-01,2,1,1.0,33.0,0.030303,0.103148,0,0,2014-01-01
5,0102,Dodoma,Mpwapwa,2014-10-01,2015-01-01,3,1,1.0,33.0,0.030303,0.103148,0,0,2014-01-01
6,0103,Dodoma,Kongwa,2014-12-01,2015-01-01,1,1,1.0,33.0,0.030303,0.074418,0,0,2014-01-01
7,0103,Dodoma,Kongwa,2014-11-01,2015-01-01,2,1,1.0,33.0,0.030303,0.074418,0,0,2014-01-01
8,0103,Dodoma,Kongwa,2014-10-01,2015-01-01,3,1,1.0,33.0,0.030303,0.074418,0,0,2014-01-01


### District-level forecast lookup

Change `DISTRICT`, `TARGET_MONTH` and `LEAD_MONTHS` to examine a specific historical issue. The calculation is shown directly from the historical counts.

In [2]:
DISTRICT = "Nanyumbu"
TARGET_MONTH = "2020-04-01"
LEAD_MONTHS = 3
selected=inspect_forecast(forecasts,DISTRICT,TARGET_MONTH,LEAD_MONTHS)
if selected.empty:
    print("No valid historical forecast for the selected district, target and lead.")
else:
    r=selected.iloc[0]
    print(f"District: {r.district} | Issue: {r.issue_month:%Y-%m} | Target: {r.target_month:%Y-%m}")
    print(f"Historical severe / valid: {r.historical_severe_years}/{r.historical_valid_years}")
    print(f"Predicted severe-drought probability: {100*r.predicted_probability:.2f}%")
    print(f"Observed SPI-3: {r.observed_spi3:.3f} | Observed severe event: {bool(r.observed_severe)}")
    display(selected)

District: Nanyumbu | Issue: 2020-01 | Target: 2020-04
Historical severe / valid: 4.0/39.0
Predicted severe-drought probability: 10.26%
Observed SPI-3: -1.769 | Observed severe event: True


,district_id,region,district,issue_month,target_month,horizon_months,calendar_month,historical_severe_years,historical_valid_years,predicted_probability,observed_spi3,observed_severe,severe_onset,latest_used_month_upper_bound
0,0905,Mtwara,Nanyumbu,2020-01-01,2020-04-01,3,4,4.0,39.0,0.102564,-1.768952,1,1,2019-04-01


### Lead-time invariance

Seasonal Climatology depends on district and target calendar month, not the lead time. For a fixed target, the one-, two- and three-month probabilities must be equal. This establishes the model's advance *risk estimate*, not an event-specific detection lead time.

In [3]:
example_all=forecasts[(forecasts.district.eq("Nanyumbu")) &
                      (forecasts.target_month.eq(pd.Timestamp("2020-04-01")))].copy()
display(example_all[["issue_month","target_month","horizon_months","historical_severe_years","historical_valid_years","predicted_probability","observed_spi3"]])
assert example_all.predicted_probability.nunique()==1
assert (forecasts.latest_used_month_upper_bound<forecasts.issue_month).all()
print("The latest included historical observation precedes every forecast issue date.")

,issue_month,target_month,horizon_months,historical_severe_years,historical_valid_years,predicted_probability,observed_spi3
28515,2020-03-01,2020-04-01,1,4.0,39.0,0.102564,-1.768952
28516,2020-02-01,2020-04-01,2,4.0,39.0,0.102564,-1.768952
28517,2020-01-01,2020-04-01,3,4.0,39.0,0.102564,-1.768952


The latest included historical observation precedes every forecast issue date.


### Retrospective case-study forecasts

The 2020–2022 subset contains 150 districts × 36 target months × 3 horizons. The same target is evaluated at each lead separately; counts across leads must not be added as independent drought events.

In [4]:
case=forecasts[forecasts.target_month.dt.year.between(2020,2022)]
print("2020–2022 total rows (three leads):",len(case))
print("Unique district-month targets:",case[["district_id","target_month"]].drop_duplicates().shape[0])
display(case[case.district.isin(["Longido","Monduli","Simanjiro"])].head(12))

2020–2022 total rows (three leads): 16200
Unique district-month targets: 5400


,district_id,region,district,issue_month,target_month,horizon_months,calendar_month,historical_severe_years,historical_valid_years,predicted_probability,observed_spi3,observed_severe,severe_onset,latest_used_month_upper_bound
27021,0201,Arusha,Monduli,2019-12-01,2020-01-01,1,1,2.0,38.0,0.052632,2.002183,0,0,2019-01-01
27022,0201,Arusha,Monduli,2019-11-01,2020-01-01,2,1,2.0,38.0,0.052632,2.002183,0,0,2019-01-01
27023,0201,Arusha,Monduli,2019-10-01,2020-01-01,3,1,2.0,38.0,0.052632,2.002183,0,0,2019-01-01
27027,0203,Arusha,Longido,2019-12-01,2020-01-01,1,1,3.0,38.0,0.078947,2.124934,0,0,2019-01-01
27028,0203,Arusha,Longido,2019-11-01,2020-01-01,2,1,3.0,38.0,0.078947,2.124934,0,0,2019-01-01
27029,0203,Arusha,Longido,2019-10-01,2020-01-01,3,1,3.0,38.0,0.078947,2.124934,0,0,2019-01-01
27348,2104,Manyara,Simanjiro,2019-12-01,2020-01-01,1,1,3.0,38.0,0.078947,1.886569,0,0,2019-01-01
27349,2104,Manyara,Simanjiro,2019-11-01,2020-01-01,2,1,3.0,38.0,0.078947,1.886569,0,0,2019-01-01
27350,2104,Manyara,Simanjiro,2019-10-01,2020-01-01,3,1,3.0,38.0,0.078947,1.886569,0,0,2019-01-01
27471,0201,Arusha,Monduli,2020-01-01,2020-02-01,1,2,3.0,38.0,0.078947,1.824257,0,0,2019-02-01
